In [11]:
from network import AudioCRNN
import torch
from network.foundation_model import Network
# from network import AngleProdict
from run import Train
model = AudioCRNN().to('cuda')
model.load_state_dict(torch.load("./experiment/ckpt/audio/model_epoch_50.pth"))
model.eval()

AudioCRNN(
  (cnn): Sequential(
    (0): Conv2d(2, 32, kernel_size=(5, 5), stride=(2, 2))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): Flatten()
    (6): Linear(in_features=2496, out_features=128, bias=True)
    (7): ReLU()
    (8): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (9): Dropout(p=0.2, inplace=False)
  )
  (attn): MultiHeadAttentionLayer(
    (fc_q): Linear(in_features=128, out_features=128, bias=True)
    (fc_k): Linear(in_features=128, out_features=128, bias=True)
    (fc_v): Linear(in_features=128, out_features=128, bias=True)
    (fc_o): Linear(in_features=128, out_features=128, bias=True)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (fnn): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (3): Linear(in_features=64, out_features=8, bia

In [20]:
import pickle
with open('val_loader.pkl' , 'rb') as f:
    loader = pickle.load(f)

In [21]:
for batch in loader:
    audio , visual , angle , action  = batch
    break

In [15]:
logits = model(audio.to('cuda'))
predict = logits.argmax(dim=1)

In [16]:
predict

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

In [17]:
action

tensor([1, 2, 1, 0, 1, 1, 1, 3, 3, 1, 1, 1, 2, 2, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 3, 1, 1, 0, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1,
        1, 0, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1,
        1, 1, 1, 1, 2, 1, 2, 2, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 3, 1, 1, 3, 1,
        3, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 0, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1,
        1, 1, 1, 3, 1, 1, 2, 0])

In [24]:
true_sector = torch.remainder(angle + 22.5, 2*180) // 45

In [25]:
true_sector

tensor([2., 2., 2., 6., 6., 1., 2., 0., 5., 2., 0., 0., 5., 0., 7., 7., 0., 0.,
        7., 0., 7., 2., 6., 6., 6., 2., 7., 0., 3., 1., 0., 2., 1., 0., 0., 2.,
        0., 1., 0., 0., 7., 0., 2., 6., 1., 0., 7., 1., 1., 0., 0., 7., 0., 7.,
        6., 1., 0., 0., 6., 0., 0., 3., 7., 0., 7., 0., 0., 6., 0., 1., 0., 2.,
        5., 7., 0., 0., 4., 0., 6., 7., 0., 7., 2., 6., 6., 1., 0., 2., 2., 0.,
        7., 0., 3., 0., 6., 0., 0., 0., 6., 2., 0., 7., 1., 2., 0., 0., 2., 2.,
        0., 0., 0., 2., 1., 0., 6., 1., 0., 7., 4., 3., 6., 6., 2., 0., 7., 7.,
        1., 0.])

In [17]:
for batch in loader:
    audio , visual , angle , action  = batch
    audio , visual , action = audio.to('cuda') , visual.to('cuda') , action.to('cuda')
    # sector_label = torch.remainder(angle + 180, 2*180) // 45
    logits = model(audio.float() , visual)
    predict = logits.argmax(dim=1)
    break

In [18]:
(predict == action).sum() / 128

tensor(0.7734, device='cuda:0')

In [19]:
predict

tensor([1, 0, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 0, 1, 1, 1, 1, 1, 1, 1, 2, 1, 0, 1, 1, 1, 1, 2, 1, 1, 1, 0, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
        1, 3, 3, 1, 1, 1, 1, 1], device='cuda:0')

In [20]:
action

tensor([3, 0, 3, 1, 3, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 3, 1, 1, 1, 1, 3, 1, 1, 3,
        1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 2, 1, 1, 1, 1,
        1, 0, 1, 1, 3, 1, 1, 1, 1, 2, 1, 0, 1, 1, 1, 1, 2, 1, 3, 2, 0, 1, 1, 1,
        1, 2, 1, 2, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1,
        1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 0, 1, 1, 1, 3, 2, 1, 1, 3, 3, 0, 1, 1, 2,
        2, 3, 1, 1, 1, 3, 1, 1], device='cuda:0')

In [17]:
predict

tensor([0, 6, 4, 4, 4, 3, 6, 3, 4, 5, 4, 4, 5, 5, 0, 6, 3, 4, 4, 3, 4, 4, 2, 4,
        4, 3, 4, 2, 3, 2, 4, 6, 2, 6, 4, 4, 4, 0, 4, 4, 4, 0, 4, 4, 4, 3, 3, 4,
        4, 0, 4, 4, 2, 4, 4, 4, 2, 4, 4, 0, 2, 4, 4, 4, 4, 2, 3, 4, 4, 4, 4, 0,
        4, 4, 3, 4, 4, 0, 0, 4, 4, 4, 4, 6, 4, 4, 5, 3, 6, 4, 0, 4, 0, 3, 0, 3,
        4, 3, 4, 4, 6, 4, 3, 4, 3, 4, 3, 6, 4, 3, 4, 5, 6, 4, 5, 4, 4, 2, 4, 6,
        4, 6, 4, 3, 5, 3, 0, 6])

In [18]:
sector_label

tensor([0., 6., 4., 4., 4., 3., 6., 3., 4., 6., 4., 4., 5., 5., 0., 6., 3., 4.,
        4., 3., 4., 4., 1., 4., 4., 3., 4., 2., 3., 2., 4., 6., 2., 6., 4., 4.,
        4., 0., 4., 4., 4., 0., 4., 4., 4., 2., 4., 4., 4., 0., 4., 4., 2., 3.,
        4., 4., 2., 4., 4., 0., 2., 4., 4., 4., 4., 2., 3., 4., 4., 3., 4., 0.,
        4., 4., 3., 4., 4., 0., 0., 4., 4., 4., 4., 6., 4., 4., 5., 2., 6., 4.,
        0., 4., 0., 3., 0., 3., 4., 3., 4., 4., 6., 4., 3., 4., 3., 4., 3., 6.,
        4., 3., 4., 5., 6., 4., 6., 4., 4., 2., 4., 6., 4., 6., 3., 3., 5., 3.,
        0., 6.])